# 03 · Build the retrieval index

Encodes every movie's composite document (`embedding_text` = title + genre + summary + plot, built by `scripts/clean_dataset.py`) with the embedding model, L2-normalises the vectors and saves them together with the metadata that the API returns.

The same notebook produces the corpus embeddings for the fine-tuned models: point `LORA_ADAPTER` at an adapter folder and give the run a `TAG`. **Query and corpus must be encoded by the same model**, so each model gets its own embeddings file; the metadata file is shared because the row order is the same.

| TAG | LORA_ADAPTER | Output |
|---|---|---|
| `mxbai_base` | `None` | `artifacts/embeddings_mxbai_base.npy` |
| `lora_v1` | `models/lora_v1` | `artifacts/embeddings_lora_v1.npy` |
| `lora_v2` | `models/lora_v2` | `artifacts/embeddings_lora_v2.npy` |

Encoding 32 K documents with the 335 M-parameter model takes a few hours on CPU.

In [ ]:
import pickle
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


In [ ]:
PROJECT_DIR = Path.cwd().parent
CSV_PATH = PROJECT_DIR / "data" / "cleaned_movie_plots_v2.csv"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

MODEL_NAME = "mixedbread-ai/mxbai-embed-large-v1"
LORA_ADAPTER = None  # e.g. PROJECT_DIR / "models" / "lora_v2"
TAG = "mxbai_base"  # -> artifacts/embeddings_{TAG}.npy
BATCH_SIZE = 16

ARTIFACTS_DIR.mkdir(exist_ok=True)


In [ ]:
df = pd.read_csv(CSV_PATH)
texts = df["embedding_text"].fillna("").tolist()
print(f"{len(df):,} movies")
df[["title", "release_year", "genre"]].head(3)


In [ ]:
model = SentenceTransformer(MODEL_NAME, device="cpu")

if LORA_ADAPTER:
    from peft import PeftModel

    transformer = model._first_module()
    transformer.auto_model = PeftModel.from_pretrained(transformer.auto_model, str(LORA_ADAPTER)).merge_and_unload()
    print(f"LoRA adapter merged from {LORA_ADAPTER}")

print(model)


In [ ]:
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype("float32")
print(embeddings.shape)


In [ ]:
np.save(ARTIFACTS_DIR / f"embeddings_{TAG}.npy", embeddings)

# The API needs everything except the full plot text.
metadata = df.drop(columns=["plot"]).to_dict(orient="records")
with open(ARTIFACTS_DIR / "metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Saved", ARTIFACTS_DIR / f"embeddings_{TAG}.npy", "and", ARTIFACTS_DIR / "metadata.pkl")


## Sanity check

Queries use mxbai's asymmetric retrieval prefix; documents do not.

In [ ]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

query = "a shark terrorizes a beach town"
vec = model.encode([f"Represent this sentence for searching relevant passages: {query}"]).astype("float32")
faiss.normalize_L2(vec)
scores, ids = index.search(vec, 5)
for score, i in zip(scores[0], ids[0]):
    print(f"{score:.3f}  {metadata[i]['title']} ({metadata[i]['release_year']})")
